# Meeting 3: generate from the training photos and train

Generates 8 pools from the 15 Egyptian Mau **training** photos and trains one run per pool, 5 seeds each, on the GPU:

- **img2img**, strength 0.3 / 0.5 / 0.7 / 0.9: the photo is the starting point; noise replaces that share of it.
- **IP-Adapter**, scale 0.3 / 0.5 / 0.7 / 1.0: the photo is a reference; the model draws a new image of a similar cat.

Every pool uses the prompts of `m2-base` (the Meeting 2 pool taken forward), so only one setting changes. Image *i* of every pool starts from the same training photo with the same seed, so pools can be compared image by image.

Storage: the pool **images** go to Google Drive (`MyDrive/dane_syntetyczne/pools/<name>/images`) and are restored from there if the runtime restarts; `model.pt` files go to Drive (`MyDrive/dane_syntetyczne/runs`); manifests, `pool.json`, contact sheets and run results go to GitHub.

Run all cells. Re-running is safe: existing images and finished seeds are skipped. Don't change the prompts or settings after the first run: a pool refuses to resume under different settings. Afterwards, compare the runs on validation in `notebooks/meeting3.ipynb` and set `IMG2IMG_RUN` and `IP_RUN` in `m3_test.ipynb`.

In [1]:
import subprocess, sys
from getpass import getpass
from pathlib import Path
BRANCH = "excercises/3"  # the branch holding colab/ and runs/
REPO = Path("/content/dane_syntetyczne")
if not (REPO / ".git").exists():
    t = getpass("GitHub token (hidden): ")
    subprocess.run(["git", "clone", "--quiet", "-b", BRANCH, f"https://{t}@github.com/tmsz/dane_syntetyczne.git", str(REPO)], check=True)
if subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", BRANCH]).returncode:
    raise RuntimeError(f"Branch {BRANCH!r} is not on GitHub. On the Mac: git push -u origin {BRANCH}")
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "--quiet", "origin", BRANCH], check=True)
sys.path.insert(0, str(REPO / "colab"))
import colab_tools as ct
import importlib
importlib.reload(ct)  # re-running this cell picks up pulled changes
ct.authenticate()  # Drive + GitHub prompts now, before any download or training

Mounted at /content/drive
✅ Google Drive mounted, GitHub push access OK


In [2]:
# The prompts and negative prompt of m2-base, the pool taken forward from Meeting 2.
# The same for every pool, so only the strength or the IP-Adapter scale changes.
BASE_PROMPTS = [
    "Egyptian Mau cat, silver coat with small dark spots, sitting on a woven rug in daylight",
    "Egyptian Mau cat lying on a fabric sofa, spotted silver fur, green eyes, indoor lighting",
    "close-up of an Egyptian Mau cat face near a window, spotted forehead with M marking, natural light",
    "bronze Egyptian Mau cat walking across a wooden floor, small spots, casual home setting",
]

NEGATIVE_PROMPT = (
    "drawing, painting, illustration, 3d render, cartoon, text, watermark, "
    "leopard, cheetah, jaguar, extra legs, deformed anatomy"
)

STRENGTHS = [0.5, 0.3, 0.7, 0.9]    # img2img: share of the photo replaced by noise
IP_SCALES = [0.5, 0.3, 0.7, 1.0]    # ip-adapter: how strongly the photo steers the image


def img2img(strength):
    return (f"m3-strength-{strength}",
            ["--mode", "img2img", "--strength", str(strength), "--negative-prompt", NEGATIVE_PROMPT],
            ["mode=img2img", f"strength={strength}"])


def ip_adapter(scale):
    return (f"m3-ip-{scale}",
            ["--mode", "ip-adapter", "--ip-adapter-scale", str(scale), "--negative-prompt", NEGATIVE_PROMPT],
            ["mode=ip-adapter", f"ip_adapter_scale={scale}"])


# Both 0.5 pools first, so they can be compared on their contact sheets early.
ORDER = [img2img(0.5), ip_adapter(0.5)] + [img2img(s) for s in STRENGTHS[1:]] + [ip_adapter(s) for s in IP_SCALES[1:]]
POOLS = {name: (options, tags) for name, options, tags in ORDER}  # name: (synth-generate options, synth-train tags)
for name, (options, tags) in POOLS.items():
    print(f"{name:18} {' '.join(options[:4]):40} tags: {', '.join(tags)}")

m3-strength-0.5    --mode img2img --strength 0.5            tags: mode=img2img, strength=0.5
m3-ip-0.5          --mode ip-adapter --ip-adapter-scale 0.5 tags: mode=ip-adapter, ip_adapter_scale=0.5
m3-strength-0.3    --mode img2img --strength 0.3            tags: mode=img2img, strength=0.3
m3-strength-0.7    --mode img2img --strength 0.7            tags: mode=img2img, strength=0.7
m3-strength-0.9    --mode img2img --strength 0.9            tags: mode=img2img, strength=0.9
m3-ip-0.3          --mode ip-adapter --ip-adapter-scale 0.3 tags: mode=ip-adapter, ip_adapter_scale=0.3
m3-ip-0.7          --mode ip-adapter --ip-adapter-scale 0.7 tags: mode=ip-adapter, ip_adapter_scale=0.7
m3-ip-1.0          --mode ip-adapter --ip-adapter-scale 1.0 tags: mode=ip-adapter, ip_adapter_scale=1.0


In [3]:
ct.setup()  # pull, install, download data, check GPU
for name, (options, tags) in POOLS.items():
    ct.restore_images(name)  # Drive -> pools/<name>/images, so a new runtime resumes the pool
    ct.generate_pool(name, BASE_PROMPTS, options)  # skips images that already exist
    ct.save_images(name)  # pool images -> Drive
    ct.save_to_github(f"Meeting 3: generate {name} in Colab")  # manifest, pool.json, contact sheet
    ct.train_run(name, "synthetic", pool=name, tags=tags)  # 5 seeds; finished seeds are skipped
    ct.check_devices([name])
    ct.save_models(name)  # model.pt -> Drive
    ct.save_to_github(f"Meeting 3: train {name} in Colab")  # everything else -> git

$ git pull --ff-only --quiet
$ /usr/bin/python3 -m pip install --quiet -e .


100%|██████████| 792M/792M [00:30<00:00, 26.1MB/s] 
100%|██████████| 19.2M/19.2M [00:01<00:00, 10.0MB/s]


Data: 905 course photos in ./data
GPU: Tesla T4
m3-strength-0.5: no images on Drive yet
GPU: Tesla T4

=== generate m3-strength-0.5 ===
$ synth-generate --name m3-strength-0.5 --device cuda --prompt Egyptian Mau cat, silver coat with small dark spots, sitting on a woven rug in daylight --prompt Egyptian Mau cat lying on a fabric sofa, spotted silver fur, green eyes, indoor lighting --prompt close-up of an Egyptian Mau cat face near a window, spotted forehead with M marking, natural light --prompt bronze Egyptian Mau cat walking across a wooden floor, small spots, casual home setting --mode img2img --strength 0.5 --negative-prompt drawing, painting, illustration, 3d render, cartoon, text, watermark, leopard, cheetah, jaguar, extra legs, deformed anatomy
Starting from 15 Egyptian Mau training photos, used in turn: Egyptian_Mau_104, Egyptian_Mau_117, Egyptian_Mau_119, Egyptian_Mau_121, Egyptian_Mau_123, Egyptian_Mau_124, Egyptian_Mau_134, Egyptian_Mau_144, Egyptian_Mau_147, Egyptian_Mau_1

### Close the Colab server

Runs only if everything above succeeded (Run all stops at the first error, so a failed session stays open to debug). Results are already on GitHub and Drive, so the runtime is released to stop using GPU quota. Set `CLOSE_SERVER = False` to keep it.

In [4]:
CLOSE_SERVER = True  # release the GPU runtime once everything is saved

if CLOSE_SERVER:
    from google.colab import runtime
    print("✅ All saved to GitHub and Drive. Closing the Colab server.")
    runtime.unassign()  # disconnects and deletes this runtime

✅ All saved to GitHub and Drive. Closing the Colab server.
